# PRCV2026 Gaussian Denoising - Restormer Fine-tune & Inference

**Team**: TinyLight (PRCV2026-0025)  
**Model**: Restormer (~26M params)  
**Rules**: Single-model, single-forward-pass, no TTA  
**GPU**: Dual T4 via DataParallel

## 0. Configuration

In [ ]:
import os

# ============ PATHS ============
KAGGLE_INPUT = "/kaggle/input/datasets/chenweijie123/prcv2026"
DATA_ROOT    = os.path.join(KAGGLE_INPUT, "data")
MODELS_ROOT  = os.path.join(KAGGLE_INPUT, "models")

TRAIN_NOISY = os.path.join(DATA_ROOT, "train", "noisy")
TRAIN_CLEAN = os.path.join(DATA_ROOT, "train", "clean")
VAL_NOISY   = os.path.join(DATA_ROOT, "val", "noisy")
PRETRAINED_WEIGHTS = os.path.join(MODELS_ROOT, "gaussian_color_denoising_sigma50.pth")

OUTPUT_DIR  = "/kaggle/working/experiments"
RESULT_DIR  = "/kaggle/working/results/val"
SUBMIT_DIR  = "/kaggle/working/submit"

# ============ TRAINING (Dual T4: 2x16GB) ============
PATCH_SIZE    = 256
BATCH_SIZE    = 4       # per-GPU batch; total = 4*2GPUs = 8
GRAD_ACCUM    = 1
LEARNING_RATE = 2e-5
TOTAL_ITERS   = 50000
WARMUP_ITERS  = 1000
EVAL_EVERY    = 5000
SAVE_EVERY    = 10000
EVAL_SAMPLES  = 20
NUM_WORKERS   = 4
USE_FP16      = True

# ============ INFERENCE ============
TILE_SIZE     = 512
TILE_OVERLAP  = 64

# ============ TEAM ============
TEAM_ID   = "PRCV2026-0025"
TEAM_NAME = "TinyLight"

for p, name in [(TRAIN_NOISY, "Train noisy"), (TRAIN_CLEAN, "Train clean"),
                (VAL_NOISY, "Val noisy"), (PRETRAINED_WEIGHTS, "Weights")]:
    print(f"{'[OK]' if os.path.exists(p) else '[MISSING]'} {name}: {p}")

In [ ]:
# If any path shows [MISSING], run this to find correct paths:
import subprocess
r = subprocess.run(['find', '/kaggle/input', '-maxdepth', '5', '-type', 'd'],
                   capture_output=True, text=True, timeout=10)
print(r.stdout)

## 1. Install & Import

In [ ]:
!pip install -q einops tqdm

import math, random, time, csv, zipfile, sys
from pathlib import Path
from tqdm.auto import tqdm

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
NUM_GPUS = torch.cuda.device_count()
print(f"GPU count: {NUM_GPUS}")
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)")

## 2. Model Architecture

Uses the **exact same attribute names** as official Restormer weights to ensure direct loading.

In [ ]:
##############################################
# Restormer - EXACT official naming scheme
# Attribute names match pretrained .pth keys
##############################################

def to_3d(x): return rearrange(x, 'b c h w -> b (h w) c')
def to_4d(x, h, w): return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.bias = nn.Parameter(torch.zeros(d))
    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        sigma = (x - mu).var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias

class LayerNorm(nn.Module):
    def __init__(self, dim, ln_type):
        super().__init__()
        self.body = BiasFree_LayerNorm(dim) if ln_type == 'BiasFree' else WithBias_LayerNorm(dim)
    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)

class FeedForward(nn.Module):
    def __init__(self, dim, ffn_factor, bias):
        super().__init__()
        hid = int(dim * ffn_factor)
        self.project_in = nn.Conv2d(dim, hid*2, 1, bias=bias)
        self.dwconv = nn.Conv2d(hid*2, hid*2, 3, padding=1, groups=hid*2, bias=bias)
        self.project_out = nn.Conv2d(hid, dim, 1, bias=bias)
    def forward(self, x):
        x = self.dwconv(self.project_in(x))
        x1, x2 = x.chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)

class Attention(nn.Module):
    def __init__(self, dim, heads, bias):
        super().__init__()
        self.num_heads = heads
        self.temperature = nn.Parameter(torch.ones(heads, 1, 1))
        self.qkv = nn.Conv2d(dim, dim*3, 1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim*3, dim*3, 3, padding=1, groups=dim*3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, 1, bias=bias)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = [rearrange(t, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
                   for t in qkv.chunk(3, dim=1)]
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        out = attn.softmax(dim=-1) @ v
        return self.project_out(rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=h, w=w))

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, ffn_factor, bias, ln_type):
        super().__init__()
        self.norm1 = LayerNorm(dim, ln_type)
        self.attn = Attention(dim, heads, bias)
        self.norm2 = LayerNorm(dim, ln_type)
        self.ffn = FeedForward(dim, ffn_factor, bias)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_c=3, dim=48, bias=False):
        super().__init__()
        self.proj = nn.Conv2d(in_c, dim, 3, padding=1, bias=bias)
    def forward(self, x): return self.proj(x)

class Downsample(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(n, n//2, 3, padding=1, bias=False), nn.PixelUnshuffle(2))
    def forward(self, x): return self.body(x)

class Upsample(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(n, n*2, 3, padding=1, bias=False), nn.PixelShuffle(2))
    def forward(self, x): return self.body(x)

class Restormer(nn.Module):
    def __init__(self, in_c=3, out_c=3, dim=48, num_blocks=None, num_refinement_blocks=4,
                 heads=None, ffn_expansion_factor=2.66, bias=False, LayerNorm_type='BiasFree'):
        super().__init__()
        if num_blocks is None: num_blocks = [4,6,6,8]
        if heads is None: heads = [1,2,4,8]
        mk = lambda d,h,n: nn.Sequential(*[TransformerBlock(d,h,ffn_expansion_factor,bias,LayerNorm_type) for _ in range(n)])

        self.patch_embed = OverlapPatchEmbed(in_c, dim)

        # NOTE: names MUST match official pretrained weights exactly
        self.encoder_level1 = mk(dim,     heads[0], num_blocks[0])
        self.down1_2        = Downsample(dim)
        self.encoder_level2 = mk(dim*2,   heads[1], num_blocks[1])
        self.down2_3        = Downsample(dim*2)
        self.encoder_level3 = mk(dim*4,   heads[2], num_blocks[2])
        self.down3_4        = Downsample(dim*4)
        self.latent         = mk(dim*8,   heads[3], num_blocks[3])

        self.up4_3              = Upsample(dim*8)
        self.reduce_chan_level3 = nn.Conv2d(dim*8, dim*4, 1, bias=bias)
        self.decoder_level3    = mk(dim*4, heads[2], num_blocks[2])

        self.up3_2              = Upsample(dim*4)
        self.reduce_chan_level2 = nn.Conv2d(dim*4, dim*2, 1, bias=bias)
        self.decoder_level2    = mk(dim*2, heads[1], num_blocks[1])

        self.up2_1          = Upsample(dim*2)
        self.decoder_level1 = mk(dim*2, heads[0], num_blocks[0])
        self.refinement     = mk(dim*2, heads[0], num_refinement_blocks)
        self.output         = nn.Conv2d(dim*2, out_c, 3, padding=1, bias=bias)

    def forward(self, inp):
        e1 = self.encoder_level1(self.patch_embed(inp))
        e2 = self.encoder_level2(self.down1_2(e1))
        e3 = self.encoder_level3(self.down2_3(e2))
        lat = self.latent(self.down3_4(e3))
        d3 = self.decoder_level3(self.reduce_chan_level3(torch.cat([self.up4_3(lat), e3], 1)))
        d2 = self.decoder_level2(self.reduce_chan_level2(torch.cat([self.up3_2(d3), e2], 1)))
        d1 = self.decoder_level1(torch.cat([self.up2_1(d2), e1], 1))
        return self.output(self.refinement(d1)) + inp

# Verify
_m = Restormer(); _n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Restormer: {_n:,} params ({_n/1e6:.2f}M) - {'[OK]' if _n<=50e6 else '[OVER LIMIT]'}")
del _m

## 3. Dataset & Utilities

In [ ]:
class DenoisingDataset(Dataset):
    def __init__(self, noisy_dir, clean_dir, patch_size=256, augment=True):
        self.patch_size, self.augment = patch_size, augment
        nd, cd = Path(noisy_dir), Path(clean_dir)
        exts = {'.png','.jpg','.jpeg','.tif','.bmp'}
        nfiles = sorted([p for p in nd.iterdir() if p.suffix.lower() in exts], key=lambda p: p.name)
        self.pairs = [(nf, cd/nf.name.replace('_noisy','_clean')) for nf in nfiles
                      if (cd/nf.name.replace('_noisy','_clean')).exists()]
        print(f"Dataset: {len(self.pairs)} pairs, patch={patch_size}")

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        nf, cf = self.pairs[idx]
        noisy = cv2.cvtColor(cv2.imread(str(nf)), cv2.COLOR_BGR2RGB)
        clean = cv2.cvtColor(cv2.imread(str(cf)), cv2.COLOR_BGR2RGB)
        h, w, ps = *noisy.shape[:2], self.patch_size
        if h < ps or w < ps:
            noisy = np.pad(noisy, ((0,max(0,ps-h)),(0,max(0,ps-w)),(0,0)), mode='reflect')
            clean = np.pad(clean, ((0,max(0,ps-h)),(0,max(0,ps-w)),(0,0)), mode='reflect')
            h, w = noisy.shape[:2]
        y, x = random.randint(0,h-ps), random.randint(0,w-ps)
        noisy, clean = noisy[y:y+ps,x:x+ps], clean[y:y+ps,x:x+ps]
        if self.augment:
            if random.random()>.5: noisy,clean = np.fliplr(noisy).copy(),np.fliplr(clean).copy()
            if random.random()>.5: noisy,clean = np.flipud(noisy).copy(),np.flipud(clean).copy()
            k = random.randint(0,3)
            if k: noisy,clean = np.rot90(noisy,k).copy(),np.rot90(clean,k).copy()
        return (torch.from_numpy(noisy.astype(np.float32)/255.).permute(2,0,1),
                torch.from_numpy(clean.astype(np.float32)/255.).permute(2,0,1))


class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps**2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred-target)**2 + self.eps2))


def calc_psnr(pred, target):
    mse = F.mse_loss(pred, target).item()
    return 100.0 if mse < 1e-10 else 10*math.log10(1.0/mse)


def build_scheduler(opt, total, warmup):
    def f(it):
        if it < warmup: return it/max(1,warmup)
        return 0.5*(1+math.cos(math.pi*(it-warmup)/max(1,total-warmup)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def load_weights(model, path):
    """Load weights, handling both official and fine-tuned checkpoints."""
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    sd = ckpt.get('params', ckpt.get('state_dict', ckpt.get('model', ckpt)))
    sd = {k.replace('module.',''):v for k,v in sd.items()}
    model.load_state_dict(sd, strict=True)
    print(f"[OK] Loaded: {path}")
    return model

print("[OK] Utilities ready.")

## 4. Setup Training (Dual GPU)

In [ ]:
device = torch.device('cuda')

# Build model & load pretrained weights
model = Restormer()
load_weights(model, PRETRAINED_WEIGHTS)
model = model.to(device)

# Wrap with DataParallel for multi-GPU
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    total_batch = BATCH_SIZE * NUM_GPUS
    print(f"[OK] DataParallel: {NUM_GPUS} GPUs, batch/gpu={BATCH_SIZE}, total_batch={total_batch}")
else:
    total_batch = BATCH_SIZE
    print(f"[INFO] Single GPU, batch={BATCH_SIZE}")

model.train()

# Dataset & Loader
dataset = DenoisingDataset(TRAIN_NOISY, TRAIN_CLEAN, patch_size=PATCH_SIZE, augment=True)
loader = DataLoader(dataset, batch_size=total_batch, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                    persistent_workers=True)

# Loss, optimizer, scheduler
criterion = CharbonnierLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = build_scheduler(optimizer, TOTAL_ITERS, WARMUP_ITERS)
scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

eff_batch = total_batch * GRAD_ACCUM
print(f"\nPatch={PATCH_SIZE}, LR={LEARNING_RATE}, FP16={USE_FP16}")
print(f"Effective batch={eff_batch}, Total iters={TOTAL_ITERS}")
print("=" * 70)

## 5. Run Training

In [ ]:
current_iter = 0
best_psnr = 0.0
running_loss = 0.0
loss_count = 0
start_time = time.time()
data_iter = iter(loader)
optimizer.zero_grad()

pbar = tqdm(total=TOTAL_ITERS, desc="Training", unit="it",
            bar_format='{l_bar}{bar:30}{r_bar}', dynamic_ncols=True)

while current_iter < TOTAL_ITERS:
    try:
        noisy, clean = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        noisy, clean = next(data_iter)

    noisy = noisy.to(device, non_blocking=True)
    clean = clean.to(device, non_blocking=True)

    with torch.amp.autocast('cuda', enabled=USE_FP16):
        pred = model(noisy)
        loss = criterion(pred, clean) / GRAD_ACCUM

    scaler.scale(loss).backward()
    running_loss += loss.item() * GRAD_ACCUM
    loss_count += 1

    if (current_iter + 1) % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.01)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

    current_iter += 1

    # Update progress bar
    if loss_count > 0:
        avg_loss = running_loss / loss_count
        lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix(loss=f"{avg_loss:.5f}", lr=f"{lr:.1e}", ordered=True)
    pbar.update(1)

    # Detailed log every 200 iters
    if current_iter % 200 == 0:
        elapsed = time.time() - start_time
        speed = current_iter / elapsed
        eta_min = (TOTAL_ITERS - current_iter) / max(speed, 1e-6) / 60
        avg_l = running_loss / max(loss_count, 1)
        lr = optimizer.param_groups[0]['lr']
        tqdm.write(f"[{current_iter:>6}/{TOTAL_ITERS}] "
                   f"loss={avg_l:.6f} lr={lr:.2e} "
                   f"speed={speed:.1f}it/s ETA={eta_min:.0f}min")
        running_loss = 0.0
        loss_count = 0

    # Evaluation
    if current_iter % EVAL_EVERY == 0:
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        raw_model.eval()
        torch.cuda.empty_cache()
        total_psnr = 0.0
        indices = random.sample(range(len(dataset)), min(EVAL_SAMPLES, len(dataset)))
        for idx in indices:
            nf, cf = dataset.pairs[idx]
            ni = cv2.cvtColor(cv2.imread(str(nf)), cv2.COLOR_BGR2RGB)
            ci = cv2.cvtColor(cv2.imread(str(cf)), cv2.COLOR_BGR2RGB)
            h, w = ni.shape[:2]
            es = 512
            if h > es or w > es:
                cy, cx = max(0,(h-es)//2), max(0,(w-es)//2)
                ni = ni[cy:cy+min(es,h), cx:cx+min(es,w)]
                ci = ci[cy:cy+min(es,h), cx:cx+min(es,w)]
            nt = torch.from_numpy(ni.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
            ct = torch.from_numpy(ci.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
            _, _, eh, ew = nt.shape
            ph, pw = (8-eh%8)%8, (8-ew%8)%8
            if ph or pw: nt = F.pad(nt, (0,pw,0,ph), mode='reflect')
            with torch.no_grad(), torch.amp.autocast('cuda', enabled=True):
                p = raw_model(nt)
            if ph or pw: p = p[:,:,:eh,:ew]
            total_psnr += calc_psnr(torch.clamp(p, 0, 1), ct)
        avg_psnr = total_psnr / len(indices)
        tqdm.write(f"  >> Eval PSNR: {avg_psnr:.4f} dB (best: {best_psnr:.4f})")
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save({'params': raw_model.state_dict()}, out_dir / 'best_model.pth')
            tqdm.write(f"  >> NEW BEST! Saved.")
        raw_model.train()
        model.train()

    # Checkpoint
    if current_iter % SAVE_EVERY == 0:
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        torch.save({'params': raw_model.state_dict(), 'iter': current_iter,
                    'best_psnr': best_psnr}, out_dir / f'ckpt_{current_iter}.pth')
        tqdm.write(f"  >> Checkpoint saved: iter {current_iter}")

pbar.close()

# Final save
raw_model = model.module if isinstance(model, nn.DataParallel) else model
torch.save({'params': raw_model.state_dict()}, out_dir / 'final_model.pth')
total_time = time.time() - start_time
print(f"\n{'='*70}")
print(f"Done! Time: {total_time/3600:.1f}h, Best PSNR: {best_psnr:.4f} dB")
print(f"Models in: {out_dir}")

## 6. Inference (Soft-blending Tiles)

In [ ]:
def tile_inference(model, inp, tile_size, tile_overlap, factor=8):
    b, c, H, W = inp.shape
    if tile_size <= 0 or (H <= tile_size and W <= tile_size):
        return model(inp)
    stride = tile_size - tile_overlap
    nx = max(1, math.ceil((W-tile_overlap)/stride))
    ny = max(1, math.ceil((H-tile_overlap)/stride))
    out = torch.zeros_like(inp)
    wm = torch.zeros(1,1,H,W, device=inp.device)
    for j in range(ny):
        for i in range(nx):
            x0 = min(i*stride, max(0,W-tile_size)); y0 = min(j*stride, max(0,H-tile_size))
            x1 = min(x0+tile_size,W); y1 = min(y0+tile_size,H)
            tile = inp[:,:,y0:y1,x0:x1]
            _,_,th,tw = tile.shape
            ph,pw = (factor-th%factor)%factor, (factor-tw%factor)%factor
            if ph or pw: tile = F.pad(tile,(0,pw,0,ph),mode='reflect')
            with torch.no_grad(): r = model(tile)
            if ph or pw: r = r[:,:,:th,:tw]
            w = torch.ones(1,1,th,tw, device=inp.device)
            if tile_overlap > 0:
                ramp = torch.linspace(0,1,tile_overlap, device=inp.device)
                if x0>0: w[:,:,:,:tile_overlap] *= ramp[None,None,None,:]
                if x1<W: w[:,:,:,-tile_overlap:] *= ramp.flip(0)[None,None,None,:]
                if y0>0: w[:,:,:tile_overlap,:] *= ramp[None,None,:,None]
                if y1<H: w[:,:,-tile_overlap:,:] *= ramp.flip(0)[None,None,:,None]
            out[:,:,y0:y1,x0:x1] += r*w
            wm[:,:,y0:y1,x0:x1] += w
    return out / wm.clamp(min=1e-8)


# Load best (or final) model
inf_model = Restormer().to(device)
best_p = Path(OUTPUT_DIR)/'best_model.pth'
final_p = Path(OUTPUT_DIR)/'final_model.pth'
w_path = str(best_p) if best_p.exists() else str(final_p)
print(f"Inference weights: {w_path}")
load_weights(inf_model, w_path)
inf_model.eval()

# Process validation set
val_dir = Path(VAL_NOISY)
res_dir = Path(RESULT_DIR); res_dir.mkdir(parents=True, exist_ok=True)
val_files = sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg'}])
print(f"Processing {len(val_files)} images (tile={TILE_SIZE}, overlap={TILE_OVERLAP})...")

total_t = 0
for i, fp in enumerate(tqdm(val_files, desc="Inference"), 1):
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    inp = torch.from_numpy(img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
    ph, pw = (8-h%8)%8, (8-w%8)%8
    if ph or pw: inp = F.pad(inp,(0,pw,0,ph),mode='reflect')
    t0 = time.time()
    with torch.amp.autocast('cuda', enabled=True):
        out_t = tile_inference(inf_model, inp, TILE_SIZE, TILE_OVERLAP)
    elapsed = time.time()-t0; total_t += elapsed
    out_t = out_t[:,:,:h,:w]
    out_np = torch.clamp(out_t,0,1)[0].permute(1,2,0).cpu().numpy()
    out_np = np.clip(np.round(out_np*255),0,255).astype(np.uint8)
    cv2.imwrite(str(res_dir/fp.name), cv2.cvtColor(out_np, cv2.COLOR_RGB2BGR))

print(f"\nDone! Total: {total_t:.0f}s, Avg: {total_t/len(val_files):.1f}s/img")

## 7. Package Submission

In [ ]:
sub_dir = Path(SUBMIT_DIR); sub_dir.mkdir(parents=True, exist_ok=True)
res_dir = Path(RESULT_DIR)
images = sorted([p for p in res_dir.iterdir() if p.suffix.lower() in {'.png','.jpg'}])
print(f"Packing {len(images)} images...")

zip_path = sub_dir / f'Results_{TEAM_NAME}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('team_info.txt', f'team_id: {TEAM_ID}\nteam_name: {TEAM_NAME}\n')
    for img in tqdm(images, desc="Zipping"):
        zf.write(img, img.name)

sz = zip_path.stat().st_size / (1024*1024)
print(f"\n[OK] {zip_path.name} ({sz:.1f} MB, {len(images)} images)")
with zipfile.ZipFile(zip_path,'r') as zf:
    ns = zf.namelist()
    print(f"  team_info.txt: {'[OK]' if 'team_info.txt' in ns else '[MISSING]'}")
    print(f"  No subdirs:    {'[OK]' if not any('/' in n for n in ns) else '[ERROR]'}")
    print(f"  Image count:   {sum(1 for n in ns if n.endswith('.png'))}")

In [ ]:
# List all output files
print("=== Output Files ===")
for root, dirs, files in os.walk('/kaggle/working'):
    level = root.replace('/kaggle/working','').count(os.sep)
    print(f"{'  '*level}{os.path.basename(root)}/")
    for f in files:
        sz = os.path.getsize(os.path.join(root,f))/(1024*1024)
        print(f"{'  '*(level+1)}{f} ({sz:.1f} MB)")